# Notebook 02_05 — Feature Selection: RFE

Runs **RFE** across K = {4, 8, 16} × 6 classifiers × 5 seeds.
Uses the pre-computed ranking from `02_00_fs_rankings.ipynb` (`models/fs_rankings.pkl`).

**Results saved incrementally to** `results/fs_rfe_raw.csv` — if this notebook crashes mid-run, just re-run it: completed (K, seed, classifier) combinations are skipped automatically.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
FEATURE_NAMES = d['feature_names']

with open(f'{MODELS_DIR}fs_rankings.pkl', 'rb') as f:
    RANKINGS = pickle.load(f)

METHOD_NAME = 'RFE'
ranked_indices = RANKINGS[METHOD_NAME]

print(f'Method: {METHOD_NAME}')
print('Top-16 features:', [FEATURE_NAMES[i] for i in ranked_indices[:16]])

Method: RFE
Top-16 features: ['tcp.dstport', 'wlan_radio.timestamp', 'tcp.srcport', 'arp.opcode', 'frame.len', 'wlan_radio.phy', 'ip.version', 'wlan.fc.subtype', 'udp.dstport', 'dns.count.queries', 'udp.srcport', 'tcp.seq', 'ip.ttl', 'wlan.seq', 'wlan.fc.ds', 'wlan.fixed.reason_code']


## Transform function (slice to top-K features by this method's ranking)

In [3]:
def transform_fn(K):
    selected = ranked_indices[:K]
    return X_train[:, selected], X_val[:, selected], X_test[:, selected], K

## Run experiment grid (resumable)

In [4]:
RESULTS_CSV = f'{RESULTS_DIR}fs_rfe_raw.csv'

run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[RFE] K=4 seed=42 DT     F1=0.5562  train=0.0s  inf=0.0001ms/sample  (1/90)
[RFE] K=4 seed=42 RF     F1=0.5566  train=1.2s  inf=0.0027ms/sample  (2/90)
[RFE] K=4 seed=42 XGB    F1=0.5566  train=1.5s  inf=0.0008ms/sample  (3/90)
[RFE] K=4 seed=42 LGBM   F1=0.5569  train=0.8s  inf=0.0014ms/sample  (4/90)
[RFE] K=4 seed=42 KNN    F1=0.3297  train=0.1s  inf=0.0381ms/sample  (5/90)
[RFE] K=4 seed=42 MLP    F1=0.5289  train=45.6s  inf=0.0017ms/sample  (6/90)
[RFE] K=4 seed=52 DT     F1=0.5565  train=0.0s  inf=0.0001ms/sample  (7/90)
[RFE] K=4 seed=52 RF     F1=0.5567  train=1.1s  inf=0.0026ms/sample  (8/90)
[RFE] K=4 seed=52 XGB    F1=0.5563  train=1.4s  inf=0.0008ms/sample  (9/90)
[RFE] K=4 seed=52 LGBM   F1=0.5569  train=0.8s  inf=0.0016ms/sample  (10/90)
[RFE] K=4 seed=52 KNN    F1=0.3296  train=0.1s  inf=0.0346ms/sample  (11/90)
[RFE] K=4 seed=52 MLP    F1=0.5307  train=52.5s  inf=0.0019ms/sample  (12/90)
[RFE] K=4 seed=62 DT     F1=0.5562  train=0.0s  inf=0.0001ms/sample  (13/90)
[RFE] 

## Quick summary

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
summary = res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std'])
print(summary.to_string())

                     f1           train_time_s            inference_ms              
                   mean       std         mean        std         mean           std
K  classifier                                                                       
4  DT          0.556299  0.000182     0.039988   0.002145     0.000054  2.193262e-06
   KNN         0.329534  0.000143     0.080244   0.003350     0.034628  4.006217e-03
   LGBM        0.556667  0.000189     0.844145   0.103722     0.001535  5.436532e-05
   MLP         0.529897  0.001757    42.122224  12.417356     0.001510  4.537069e-04
   RF          0.556408  0.000238     1.142499   0.016329     0.002675  1.653017e-05
   XGB         0.556640  0.000380     1.483438   0.049177     0.000831  1.605058e-05
8  DT          0.889124  0.000659     0.054315   0.001440     0.000056  1.197879e-06
   KNN         0.739385  0.006381     0.147487   0.001370     0.022140  3.209840e-03
   LGBM        0.889106  0.000586     1.222418   0.061032     0.0